# FGBA model: Forest fires

## Useful resources:
- Paper (Wuyts and Sieber 2024): https://doi.org/10.1073/pnas.2211853120
- Agents.jl documentation: https://juliadynamics.github.io/Agents.jl/stable/
- Agents.jl example: https://juliadynamics.github.io/AgentsExampleZoo.jl/dev/examples/forest_fire/

In [ ]:
# install needed packages in local environment
using Pkg
Pkg.activate(".")
Pkg.add("Agents")
Pkg.add("CairoMakie")
Pkg.add("IJulia")
Pkg.add("Observables")
Pkg.add("Random")

using CairoMakie
using DataFrames
using Observables

# Load model source code
include("fgba_model.jl");

In [ ]:
# Model parameters
p = FGBA_parameters()

# Print all model parameter values
for name in fieldnames(typeof(p))
    println(name, " = ", getfield(p, name))
end

In [ ]:
# Build model
fgba = fgba_model(p)

In [ ]:
# Run simulation for t_end years
t_end = 1000.0
_, data = run!(fgba, (m, s) -> m.t[] >= t_end; mdata, when = (m, s) -> s % 200 == 0)

# Print data sample
data[1:2, 1:end-1]

In [ ]:
# Save simulation result as GIF and show
state_obs = Observable(data.state_map[1])
fig_anim = Figure(size = (1100, 500))
ax1_anim = Axis(fig_anim[1, 1]; title = "Spatial domain", aspect = DataAspect())
heatmap!(
    ax1_anim,
    state_obs;
    colorrange = (1, 4),
    colormap = cgrad([clrF, clrG, clrB, clrA]; categorical = true)
)
hidedecorations!(ax1_anim)
ax2_anim = Axis(
    fig_anim[1, 2];
    xlabel = "Time",
    ylabel = "Area fraction",
    title = "Aggregated timeseries",
)

time_obs   = Observable(data.time[1:1])
forest_obs = Observable(data.forest_frac[1:1])
grass_obs  = Observable(data.grass_frac[1:1])
burn_obs   = Observable(data.burn_frac[1:1])
ash_obs    = Observable(data.ash_frac[1:1])

lines!(ax2_anim, time_obs, forest_obs; color = clrF, label = "forest")
lines!(ax2_anim, time_obs, grass_obs;  color = clrG, label = "grass")
lines!(ax2_anim, time_obs, burn_obs;   color = clrB, label = "burning")
lines!(ax2_anim, time_obs, ash_obs;    color = clrA, label = "ash")
axislegend(ax2_anim)

# save GIF
gif_path = "forest_fire.gif"
time_step = 2
record(fig_anim, gif_path, 1:time_step:nrow(data); framerate = 20) do i
    state_obs[] = data.state_map[i]
    time_obs[]   = data.time[1:i]
    forest_obs[] = data.forest_frac[1:i]
    grass_obs[]  = data.grass_frac[1:i]
    burn_obs[]   = data.burn_frac[1:i]
    ash_obs[]    = data.ash_frac[1:i]
    xlims!(ax2_anim, 0, max(1, data.time[i]))
end

play_gif("forest_fire.gif")